In [149]:
import numpy as np

# Tensor operations return lazybuffers, if you realize the lazybuffers, they run forward()

### SHAPETRACKER ###

class ShapeTracker:
    def __init__(self, shape) -> None:
        self.shape = shape 
        self.permutes = None
        self.stride = [1]
        self.pad = [0]

### LAZYBUFFER ###

class LazyBuffer:
    def __init__(self) -> None:
        self.buffer = []
        self.grad = 0
    
    def numpy(self):
        return self.forward()
    
    def debug(self):
        print(self.buffer)
    
    def __add__(self, other): return Add(self, other)
    def __mul__(self, other): return Mul(self, other)
    def __truediv__(self, other): return Div(self, other)
    def __rtruediv__(self, other): return Div(other, self) # Really untested
    def __sub__(self, other): return Add(self, -other)
    def __matmul__(self, other): return MatMul(self, other)
    def __neg__(self): return Mul(self, -1)
    def __pow__(self, exp): return Pow(self, exp)
    def __exp__(self): return Exp(self)

    def sum(self, axis=None): return Sum(self, axis)
    def mean(self, axis=None): return Sum(self, axis) * (1 / self.shapeTrack.shape[axis])  # Untested
    def onehot(self, dict_size=None): return OneHot(self, dict_size) # NOT LAZY
    def softmax(self): return softmax(self)  # Untested
    def max(self, axis=None): return Max(self, axis)  # Untested

    def __str__(self): return self.numpy().__str__()


### TENSOR ###

class Tensor(LazyBuffer):  # Tensor is just lazybuffer that contains data 
    def __init__(self, value) -> None:
        self.data = np.array(value)
        self.shape = self.data.shape
        self.shapeTrack = ShapeTracker(self.shape)
        self.requires_grad = False
        self.grad = 0

    def forward(self):
        return self.data
    def backward(self, grad):  # Leaf tensors don't need to do anything
        self.grad += grad

    def numpy(self):
        return self.data

    def __getitem__(self, index):   # THIS IS NOT LAZY!!
        if isinstance(index, slice):
            start, stop, step = index.indices(len(self.data))
            return Tensor(self.data[start:stop:step])
        else:
            return Tensor(self.data[index])


### OPERATION TYPES ###

class Unary(LazyBuffer):
    def __init__(self, a) -> None:
        super().__init__()
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a


class Binary(LazyBuffer):
    def __init__(self, a, b) -> None:
        super().__init__()
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a
        self.b = Tensor(b) if not isinstance(b, LazyBuffer) else b

class Reduce(LazyBuffer):
    def __init__(self, a, axis=None) -> None:
        super().__init__()
        self.grad = 0
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a
        self.axis = axis
        # self.shape = 1 if axis is None else (elm for i, elm in enumerate(self.a.shape) if i != axis)

class Broadcast(LazyBuffer):
    def __init__(self, a, b) -> None:
        super().__init__()
        self.a = Tensor(a) if not isinstance(a, LazyBuffer) else a
        self.b = Tensor(b) if not isinstance(b, LazyBuffer) else b

### OPERATIONS ###

def softmax(tensor): # safe softmax
    exp = Exp(tensor - tensor.max())
    return exp / Repeat(exp.sum(axis=-1), 10)  # 10 is extremely hard coded for testing (its late im tired)

def OneHot(tensor, dict_size):
    if dict_size is None: dict_size = int(tensor.data.max()) + 1
    result = np.zeros((tensor.shape[0], dict_size))
    result[np.arange(tensor.shape[0]), np.int32(tensor.data)] = 1
    return Tensor(result)    

class Exp(Unary):
    def forward(self):
        self.data = np.exp(self.a.forward())
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * self.data)

class Div(Binary):
    def forward(self):
        self.data = self.a.forward() / self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad / self.b.data)
        self.b.backward(grad * -self.a.data / (self.b.data ** 2))

class Pow(Binary):
    def forward(self):  # a ** b
        self.data = self.a.forward() ** self.b.forward()  
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * self.b.data * self.a.data ** (self.b.data - 1))
        self.b.backward(grad * self.data * np.log(self.a.data))

class Add(Binary):
    def forward(self):
        self.data = self.a.forward() + self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad)
        self.b.backward(grad)

class Mul(Binary):
    def forward(self):
        self.data = self.a.forward() * self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * self.b.data)
        self.b.backward(grad * self.a.data)

class MatMul(Binary):
    def forward(self):
        self.data = self.a.forward() @ self.b.forward()
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad @ self.b.data.T)
        self.b.backward(self.a.data.T @ grad)


class Sum(Reduce):
    def forward(self):
        self.data = np.sum(self.a.forward(), axis=self.axis)
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(np.repeat(grad, 10, axis=self.axis).reshape(*self.a.shape)) # Untested
                                  
class Mean(Reduce):  # Untested
    def forward(self):
        self.data = np.mean(self.a.forward(), axis=self.axis)
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(grad * np.ones(self.a.shape) * (1 / self.a.shapeTrack.shape[self.axis]))

class Max(Reduce): 
    def forward(self):
        self.data = np.max(self.a.forward(), axis=self.axis)
        self.shape = self.data.shape
        return self.data
    
    def backward(self, grad=1):  # WTF is the gradient of max? Definetly untested
        self.grad += grad
        self.a.backward(grad * (self.a.data == self.data))

class Repeat(Broadcast):
    def forward(self):
        self.data = np.repeat(self.a.forward(), self.b.forward(), axis=-1).reshape(*self.a.forward().shape, self.b.forward())
        self.shape = self.data.shape
        return self.data

    def backward(self, grad=1):
        self.grad += grad
        self.a.backward(np.sum(grad, axis=-1))
        self.b.backward(np.sum(grad * self.a.data, axis=-1))


class Rand(Tensor):
    def __init__(self, shape) -> None:
        self.data = np.random.rand(*shape)
        super().__init__(self.data)

class Randn(Tensor):
    def __init__(self, shape) -> None:
        self.data = np.random.randn(*shape)
        super().__init__(self.data)

class ReadCSV(Tensor):
    def __init__(self, path) -> None:
        # assume first line is the header
        self.data = np.genfromtxt(path, delimiter=',', skip_header=1)
        super().__init__(self.data)


### MLOPS ###

class Convolution(LazyBuffer):
    def forward(x, weight, bias, stride, padding):
        pass

### MODULES ###

class Module():
    pass

class Linear(Module):
    def __init__(self, in_features, out_features) -> None:
        self.in_features = in_features
        self.out_features = out_features
        self.weight = Randn((in_features, out_features)) * (1 / np.sqrt(in_features))
        self.bias = Randn((1, out_features)) * (1 / np.sqrt(in_features)) # double check the 1 in (1, out_features)

    def forward(self, x):
        self.x = x
        return x @ self.weight + self.bias

    def backward(self, grad):
        self.weight.grad += self.x.T @ grad
        self.bias.grad += grad.sum(axis=0)
        return grad @ self.weight.T
    
class Conv2D(Module): # Not yet supported 
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0) -> None:
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.weight = Rand((out_channels, in_channels, kernel_size, kernel_size))
        self.bias = Rand((1, out_channels))

    def forward(self, x):  # Change this, define convolution as reshaped matmul
        assert len(x.shape) == 4, "Input tensor must be 4D (batch, channel, height, width)"
        # result = np.zeros((x.shape[0], self.out_channels, *x.shape[2:]))  
        
        # skip = int(self.kernel_size/2)
        # for i in range(skip, x.shape[2]-skip, self.stride):
        #     for j in range(skip, x.shape[3]-skip, self.stride):
        #         result[:, :, i, j] = x.data[:, :, i-skip:i+skip, j-skip:j+skip] * self.weight

        # return Tensor(result + self.bias)
        return Convolution.forward(x, self.weight, self.bias, self.stride, self.padding)

    def backward(self, grad):
        pass  # Can't be bothered 



        


In [118]:
# would be cool to run on MNIST

dataset = ReadCSV('mnist_train.csv')


KeyboardInterrupt: 

In [150]:
images = dataset[:, 1:]
labels = dataset[:, 0]

fc1 = Linear(784, 10)
mse = ((fc1.forward(images).softmax() + (- labels.onehot())) ** 2).sum()
print(mse.numpy())
mse.backward()

109813.15396756928


ValueError: cannot reshape array of size 10 into shape (60000,10)

In [133]:
# Softmax testing
n = Rand((1, 10))
print(Exp(n))
print(Exp(n).sum(1))
print(n.softmax())

[[2.32357242 1.29060336 1.17901193 1.04738572 2.06071516 2.01357728
  1.07164179 2.26901126 2.6726729  1.94057902]]
[17.86877084]
[[0.13003538 0.07222676 0.0659817  0.05861543 0.11532495 0.11268695
  0.05997289 0.12698194 0.14957229 0.10860171]]


In [14]:
output.numpy()

array([[ 117.61452441, -111.65694779,  107.36385164, ...,   53.58838391,
         106.69311709,  -83.32522846],
       [  89.56216287,  -13.8187122 ,  107.26697021, ...,   73.73735931,
         117.12209774,  -11.48190668],
       [  44.85184525,   25.15065779,  -37.71456661, ...,   40.05536546,
         -36.57610678, -104.04146739],
       ...,
       [ 141.54043248,    9.65783841,   -1.56395171, ...,  163.73343039,
         -74.13833273,  -36.1691029 ],
       [  25.13424445,   18.97431544,  107.04201011, ...,   23.80922902,
          28.84876724, -122.98191804],
       [  76.78514152,  -18.86484713,   68.92965517, ...,  -11.26740982,
          64.50984431,  -91.08108858]])

In [2]:
Linear(3, 2).forward(Rand((2, 3))).numpy()

array([[1.4965278 , 1.49941769],
       [1.75843499, 1.47767739]])

In [3]:
conv = Conv2D(3, 2, 3)
conv.forward(Rand((2, 3, 4, 4)))

In [5]:
a = Tensor(np.array([[1, 2], [3, 4]]))
b = Tensor(np.array([[5, 6], [7, 8]]))

c = a + b
d = c * b
e = d + 3
f = e.sum()

print(c)
print(d)
print(e)
print(f)

print(f.numpy())
f.backward()

print(a.grad)
print(b.grad)
print(c.grad)
print(d.grad)
print(e.grad)
print(f.grad)


256
[[5. 6.]
 [7. 8.]]
[[11. 14.]
 [17. 20.]]
[[5. 6.]
 [7. 8.]]
[[1. 1.]
 [1. 1.]]
[[1. 1.]
 [1. 1.]]
1


In [6]:
from torch import *

In [7]:
a = Tensor(np.array([[1, 2], [3, 4]])).requires_grad_()
b = Tensor(np.array([[5, 6], [7, 8]])).requires_grad_()

c = a + b
d = c * b
e = d + 3
f = e.sum()

print(c)
print(d)
print(e)
print(f)

f.backward()

print("-------------------")
print(a.grad)
print(b.grad)
# print(c.grad)
# print(d.grad)
# print(e.grad)
# print(f.grad)


tensor([[ 6.,  8.],
        [10., 12.]], grad_fn=<AddBackward0>)
tensor([[30., 48.],
        [70., 96.]], grad_fn=<MulBackward0>)
tensor([[33., 51.],
        [73., 99.]], grad_fn=<AddBackward0>)
tensor(256., grad_fn=<SumBackward0>)
-------------------
tensor([[5., 6.],
        [7., 8.]])
tensor([[11., 14.],
        [17., 20.]])
